# DRO-FAIR Analysis Notebook

Interactive analysis of DRO-FAIR experiment results.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load results
with open('../results/all_results.json', 'r') as f:
    results = json.load(f)

print(f"Total experiments: {len(results)}")

In [ ]:
# Create DataFrame for easy analysis
rows = []
for r in results:
    for method in ['naive', 'dro']:
        rows.append({
            'dataset': r['dataset'],
            'alpha': r['alpha'],
            'seed': r['seed'],
            'method': method.upper(),
            'accuracy': r[method]['accuracy'],
            'dp_violation': r[method]['dp_violation'],
            'if_violation': r[method]['if_violation'],
        })

df = pd.DataFrame(rows)
df.head()

In [ ]:
# Summary statistics by dataset and alpha
summary = df.groupby(['dataset', 'alpha', 'method']).agg({
    'accuracy': ['mean', 'std'],
    'dp_violation': ['mean', 'std'],
    'if_violation': ['mean', 'std']
}).round(4)

summary

In [ ]:
# Plot accuracy vs corruption level
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, dataset in enumerate(['adult', 'credit', 'lsac']):
    ax = axes[idx]
    subset = df[df['dataset'] == dataset]
    
    for method, color in [('NAIVE', 'C0'), ('DRO', 'C1')]:
        data = subset[subset['method'] == method]
        grouped = data.groupby('alpha')['accuracy'].agg(['mean', 'std'])
        alphas = grouped.index
        means = grouped['mean']
        stds = grouped['std']
        
        ax.errorbar(alphas, means, yerr=stds/np.sqrt(10), 
                   label=method, marker='o', color=color)
    
    ax.set_xlabel('Corruption Rate (α)')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{dataset.upper()}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot DP violation vs corruption level
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, dataset in enumerate(['adult', 'credit', 'lsac']):
    ax = axes[idx]
    subset = df[df['dataset'] == dataset]
    
    for method, color in [('NAIVE', 'C0'), ('DRO', 'C1')]:
        data = subset[subset['method'] == method]
        grouped = data.groupby('alpha')['dp_violation'].agg(['mean', 'std'])
        alphas = grouped.index
        means = grouped['mean']
        stds = grouped['std']
        
        ax.errorbar(alphas, means, yerr=stds/np.sqrt(10), 
                   label=method, marker='s', color=color)
    
    ax.set_xlabel('Corruption Rate (α)')
    ax.set_ylabel('DP Violation')
    ax.set_title(f'{dataset.upper()}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compute improvement percentages
improvements = []
for dataset in ['adult', 'credit', 'lsac']:
    for alpha in [0.1, 0.2, 0.3]:
        subset = df[(df['dataset'] == dataset) & (df['alpha'] == alpha)]
        
        naive_dp = subset[subset['method'] == 'NAIVE']['dp_violation'].mean()
        dro_dp = subset[subset['method'] == 'DRO']['dp_violation'].mean()
        naive_if = subset[subset['method'] == 'NAIVE']['if_violation'].mean()
        dro_if = subset[subset['method'] == 'DRO']['if_violation'].mean()
        naive_acc = subset[subset['method'] == 'NAIVE']['accuracy'].mean()
        dro_acc = subset[subset['method'] == 'DRO']['accuracy'].mean()
        
        dp_reduction = (naive_dp - dro_dp) / (naive_dp + 1e-8) * 100
        if_reduction = (naive_if - dro_if) / (naive_if + 1e-8) * 100
        acc_change = (dro_acc - naive_acc) * 100
        
        improvements.append({
            'dataset': dataset,
            'alpha': alpha,
            'dp_reduction_%': dp_reduction,
            'if_reduction_%': if_reduction,
            'accuracy_change_%': acc_change
        })

imp_df = pd.DataFrame(improvements)
print(imp_df.to_string(index=False))